# Un agente ReAct mínimo, paso a paso

**Facsímil 5 · Agentes y orquestación** — capítulos 2, 3 y 5
(qué es un agente: estado, acción y observación; *tools*; de ReAct a sistemas multiagente).

Un agente no es «un prompt más listo». Es un **bucle** alrededor del modelo: decidir una acción,
ejecutarla, mirar el resultado y volver a decidir, hasta resolver la tarea. En este cuaderno construyes
el esqueleto de ese bucle —el patrón **ReAct** (*Reason + Act*)— y lo ves resolver una tarea que no se
puede contestar de un tirón, usando herramientas de verdad y dejando una **traza** auditable de su
razonamiento. El «modelo» que decide la siguiente acción lo simulamos con reglas, para que el bucle
—que es lo importante— se vea sin caja negra. Entender ese ciclo es entender por qué un agente puede
encadenar pasos... y por qué a veces se va por las ramas.

### Qué vas a aprender
- Qué distingue a un **agente** de una sola llamada a un modelo.
- El bucle **ReAct**: pensar → actuar → observar, repetido hasta tener la respuesta.
- Qué son las **herramientas** (*tools*) y cómo el agente las usa para tocar el mundo.
- Por qué los **límites** (tope de pasos, validación, traza) no son un detalle, sino lo que separa un
  agente útil de uno que se descontrola.

### Cuánto cuesta
Unos 12 minutos. CPU, sin claves (el «modelo» se simula con reglas).


> **Inteligencia artificial para gente curiosa** · facsímil interactivo
> 
> Web del facsímil: https://www.iaparagentecuriosa.686f6c61.dev/ · Autor: @686f6c61 · Fecha: 2026-06-26 · Versión 1.0
> 
> Este cuaderno acompaña al facsímil: ejecútalo de arriba abajo, lee cada celda de texto
> antes de correr la de código y detente en las salidas. La gracia no es que «salga», sino
> entender *por qué* sale.

In [1]:
import ast, operator
# Las herramientas: funciones normales que el agente puede invocar.
CATALOGO = {"teclado": 29.90, "raton": 15.50, "monitor": 149.00, "webcam": 39.00}
PEDIDOS  = {"1002": "en reparto, llega manana", "1003": "entregado el 24/05"}

# Calculadora SEGURA con ast (nunca eval, que ejecutaria codigo arbitrario).
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub,
        ast.Mult: operator.mul, ast.Div: operator.truediv, ast.USub: operator.neg}
def _ev(n):
    if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
    if isinstance(n, ast.BinOp) and type(n.op) in _OPS: return _OPS[type(n.op)](_ev(n.left), _ev(n.right))
    if isinstance(n, ast.UnaryOp) and type(n.op) in _OPS: return _OPS[type(n.op)](_ev(n.operand))
    raise ValueError("expresion no permitida")
def calculadora(expr): return _ev(ast.parse(expr, mode="eval").body)
def buscar_precio(p): return CATALOGO.get(p.lower(), "no encontrado")
def estado_pedido(n): return PEDIDOS.get(str(n), "pedido desconocido")

TOOLS = {"calculadora": calculadora, "buscar_precio": buscar_precio, "estado_pedido": estado_pedido}
print("Herramientas disponibles:", ", ".join(TOOLS))


Herramientas disponibles: calculadora, buscar_precio, estado_pedido


## 1. Agente o prompt: cuándo merece la pena

Para muchas tareas, una sola llamada al modelo basta: «resume este texto», «traduce esta frase». Pero
otras requieren **varios pasos que dependen unos de otros** y **tocar el mundo** (buscar un dato,
calcular, consultar un sistema). Ahí entra un agente: un bucle que decide qué hacer, lo hace, mira el
resultado y sigue. La regla práctica del capítulo 1: usa un agente solo cuando la tarea sea multi-paso,
no se pueda resolver de un tirón, y el coste de la complejidad compense. Nuestra tarea de ejemplo es
justo así: «quiero 3 teclados, cuánto es y cómo va mi pedido 1002» mezcla un cálculo, una búsqueda y
una consulta.


## 2. El bucle ReAct: pensar → actuar → observar

ReAct alterna **razonar** y **actuar**. En cada vuelta, el agente escribe un **pensamiento** (qué le
falta), elige una **acción** (qué herramienta y con qué argumento), y recibe una **observación** (el
resultado). Repite hasta tener todo para responder. Aquí el «cerebro» que decide la siguiente acción es
una función con reglas; en un agente real, esa decisión la toma el LLM, pero **el bucle es idéntico**.


In [2]:
def cerebro(memoria):
    # Decide la siguiente accion segun lo que ya sabe (memoria). Determinista.
    hechos = dict(memoria)
    if "precio_teclado" not in hechos:
        return ("buscar_precio", "teclado", "Necesito el precio del teclado")
    if "total" not in hechos:
        return ("calculadora", f"3 * {hechos['precio_teclado']}", "Calculo 3 teclados")
    if "pedido" not in hechos:
        return ("estado_pedido", "1002", "Consulto el estado del pedido 1002")
    return ("RESPONDER", None, "Ya tengo precio, total y estado")

def agente(tarea, max_pasos=8):
    memoria = []
    print(f"TAREA: {tarea}\n")
    for paso in range(1, max_pasos+1):
        accion, arg, pensamiento = cerebro(memoria)
        print(f"Paso {paso}")
        print(f"  Pensamiento: {pensamiento}")
        if accion == "RESPONDER":
            d = dict(memoria)
            print(f"  RESPUESTA: tres teclados cuestan {d['total']:.2f} EUR; tu pedido esta {d['pedido']}.")
            return
        obs = TOOLS[accion](arg)
        print(f"  Accion: {accion}({arg!r})")
        print(f"  Observacion: {obs}")
        clave = {"buscar_precio":"precio_teclado", "calculadora":"total", "estado_pedido":"pedido"}[accion]
        memoria.append((clave, obs)); print()

agente("Quiero 3 teclados, cuanto es y como va mi pedido 1002?")


TAREA: Quiero 3 teclados, cuanto es y como va mi pedido 1002?

Paso 1
  Pensamiento: Necesito el precio del teclado
  Accion: buscar_precio('teclado')
  Observacion: 29.9

Paso 2
  Pensamiento: Calculo 3 teclados
  Accion: calculadora('3 * 29.9')
  Observacion: 89.69999999999999

Paso 3
  Pensamiento: Consulto el estado del pedido 1002
  Accion: estado_pedido('1002')
  Observacion: en reparto, llega manana

Paso 4
  Pensamiento: Ya tengo precio, total y estado
  RESPUESTA: tres teclados cuestan 89.70 EUR; tu pedido esta en reparto, llega manana.


**Mira la traza.** El agente no contestó de golpe: descompuso la tarea en pasos, usó una herramienta
distinta en cada uno y fue **acumulando observaciones** hasta poder responder. Ese ir y venir —decidir,
ejecutar, mirar— es exactamente lo que hace un agente real; lo único que cambia es que ahí el LLM elige
la acción en vez de nuestras reglas. Y la traza (pensamiento/acción/observación) hace cada paso
**auditable**: puedes ver *por qué* hizo lo que hizo.


## 3. Por qué esto no es un juguete: los límites

El bucle tiene un riesgo real. Si el «cerebro» se equivoca al elegir la acción, el agente puede quedarse
**dando vueltas** o llamar a la herramienta equivocada gastando tiempo y dinero sin avanzar. Por eso los
agentes serios ponen barreras: un **tope de pasos**, validación de cada acción, y la traza para
diagnosticar. Simulamos un cerebro roto que nunca termina y vemos cómo el tope lo corta.


In [3]:
def cerebro_roto(memoria): return ("buscar_precio", "teclado", "...otra vez lo mismo, no aprendo")
pasos = 0; mem = []
TOPE = 5
while pasos < TOPE:                      # SIN este tope, seria un bucle infinito
    a, arg, _ = cerebro_roto(mem); pasos += 1
    if a == "RESPONDER": break
print(f"El agente con cerebro roto se corto a los {pasos} pasos gracias al tope.")
print("Sin ese limite, habria llamado a la herramienta para siempre, gastando dinero.")
print("El tope de pasos es la red de seguridad mas basica (y mas importante) de todo agente.")


El agente con cerebro roto se corto a los 5 pasos gracias al tope.
Sin ese limite, habria llamado a la herramienta para siempre, gastando dinero.
El tope de pasos es la red de seguridad mas basica (y mas importante) de todo agente.


## 4. Reaccionar a las malas observaciones

Un agente bueno no solo encadena pasos: **reacciona** cuando una herramienta devuelve algo inesperado.
Si el catálogo no tiene el producto, la observación es «no encontrado», y el agente debería cambiar de
plan en vez de seguir como si nada. Lo vemos con un cerebro que detecta el problema y responde con
honestidad en vez de inventarse un precio.


In [4]:
def cerebro_robusto(memoria):
    hechos = dict(memoria)
    if "precio" not in hechos:
        return ("buscar_precio", "altavoz", "Busco el precio del altavoz")   # no esta en el catalogo
    return ("RESPONDER", None, "Decido que hacer con lo que observe")

mem = []
for paso in range(1, 4):
    accion, arg, pensamiento = cerebro_robusto(mem)
    if accion == "RESPONDER":
        precio = dict(mem).get("precio")
        if precio == "no encontrado":
            print("RESPUESTA: lo siento, ese producto no esta en el catalogo. (No me invento un precio.)")
        break
    obs = TOOLS[accion](arg); print(f"Paso {paso}: {accion}({arg!r}) -> {obs}")
    mem.append(("precio", obs))


Paso 1: buscar_precio('altavoz') -> no encontrado
RESPUESTA: lo siento, ese producto no esta en el catalogo. (No me invento un precio.)


**Eso es robustez.** El agente preguntó por un producto que no existe, recibió «no encontrado» y, en
vez de alucinar un precio, lo admitió. Un agente que ignora las malas observaciones es peligroso: actúa
sobre datos que no tiene. Reaccionar a lo que el mundo te devuelve es parte del oficio.


## 5. Pruébalo tú

1. **Añade una herramienta** `tiempo(ciudad)` con un diccionario y una tarea que la necesite. Amplía el
   `cerebro` para usarla. ¿Ves cómo el bucle no cambia, solo las acciones disponibles?
2. **Baja el tope** a `max_pasos=2`. El agente se queda sin pasos antes de responder. Así se ve por qué
   el límite es un equilibrio entre seguridad y capacidad.
3. **Encadena más tareas:** «cuánto cuestan 2 monitores y 1 webcam». ¿Cuántos pasos necesita ahora?
4. **Registra la traza en una lista** en vez de imprimirla: eso es, en pequeño, la observabilidad de un
   agente real (facsímil 6).


## 6. Errores comunes

- **Pensar que un agente es magia.** Es un bucle con herramientas. La inteligencia está tanto en el
  modelo como en el **andamiaje** (las tools, los límites, la traza).
- **No poner tope de pasos.** Un agente sin freno puede entrar en bucle y gastar tiempo y dinero sin
  avanzar. El tope es innegociable.
- **Ignorar las malas observaciones.** Si una herramienta falla y el agente sigue como si nada, actúa
  sobre datos inventados.
- **No dejar traza.** Sin registro de pensamiento/acción/observación, es imposible depurar por qué el
  agente hizo lo que hizo.


## 7. Qué te llevas

- Un agente es un **bucle** (pensar, actuar, observar), no un prompt mágico. La inteligencia está tanto
  en el modelo como en el andamiaje que lo rodea.
- Las **herramientas** son funciones normales con las que el agente toca el mundo; la **traza** hace
  auditable cada paso.
- Los **límites** (tope de pasos, validación, reacción a errores) son lo que separa un agente útil de
  uno que se descontrola.

**Para seguir:** el siguiente cuaderno formaliza cómo el modelo «pide» una herramienta (*function
calling*) y cómo validas esa petición antes de ejecutar; el capítulo 10 mide la trayectoria completa de
un agente; y el facsímil 6 lleva la traza a la observabilidad en producción.


---

### Ficha del cuaderno

- **Obra:** *Inteligencia artificial para gente curiosa* (facsímil interactivo).
- **Web:** https://www.iaparagentecuriosa.686f6c61.dev/
- **Autor:** @686f6c61
- **Fecha:** 2026-06-26
- **Versión:** 1.0

*Material pedagógico. Las salidas que ves son reales: se generan al ejecutar el código, no están escritas a mano. Si cambias algo, cambiarán: esa es la idea.*